# 🔧 Notebook 2: Signal Preprocessing & Data Pipeline

## CWRU Bearing Fault Detection System

---

### Objectives
1. Understand and visualize each preprocessing step
2. Demonstrate the bandpass filter's effect on signals
3. Compare normalization methods (Z-score, Min-Max, MaxAbs)
4. Analyze windowing with different overlap strategies
5. Compare data splitting methods (Time-based, File-based, Hybrid)
6. Verify no data leakage exists between train/test sets
7. Examine data augmentation techniques

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch, freqz, butter
from scipy.stats import kurtosis
from collections import Counter
import torch
from torch.utils.data import DataLoader
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import CWRUDataLoader
from src.data.preprocessing import (
    bandpass_filter, normalize_signal, create_windows,
    time_based_split, hybrid_split, file_based_split,
    PreprocessingPipeline
)
from src.training.train import BearingDataset

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 100})

CLASS_LABELS = CWRUDataLoader.CLASS_LABELS
FS = 12000

print("✅ Environment ready!")

In [ ]:
# Load dataset
loader = CWRUDataLoader(data_dir=str(PROJECT_ROOT / 'data' / 'cwru'))
signals_dict, metadata_df = loader.load_all_data()
labels_dict = loader.get_labels_dict(metadata_df)

# Get representative signals
representative = {}
for fname, cid in loader.FILE_MAPPING.items():
    if cid not in representative and fname in signals_dict:
        representative[cid] = fname

print(f"Loaded {len(signals_dict)} files")

## 2. Bandpass Filter Analysis

The bandpass filter (Butterworth, 4th order, 10–5000 Hz) removes:
- **DC offset** and very low-frequency drift
- **High-frequency noise** above the Nyquist-relevant range

In [ ]:
# ============================================================
# Filter frequency response
# ============================================================
nyquist = 0.5 * FS
low = 10.0 / nyquist
high = min(5000.0 / nyquist, 0.99)
b, a = butter(4, [low, high], btype='band')
w, h = freqz(b, a, worN=8000, fs=FS)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Magnitude response
axes[0].plot(w, 20 * np.log10(np.abs(h)), 'b', linewidth=2)
axes[0].axvline(10, color='red', linestyle='--', alpha=0.7, label='Lowcut (10 Hz)')
axes[0].axvline(5000, color='red', linestyle='--', alpha=0.7, label='Highcut (5000 Hz)')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Magnitude (dB)')
axes[0].set_title('Bandpass Filter — Frequency Response', fontweight='bold')
axes[0].set_ylim([-80, 5])
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Phase response
axes[1].plot(w, np.unwrap(np.angle(h)) * 180 / np.pi, 'g', linewidth=2)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Phase (degrees)')
axes[1].set_title('Phase Response (filtfilt → zero phase)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📝 Note: Using scipy.signal.filtfilt (zero-phase filtering)")
print("   → No phase distortion in the output signal")

In [ ]:
# ============================================================
# Before vs After filtering
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

example_classes = [0, 1, 4, 7]
titles = ['Normal', 'Inner Race 007', 'Outer Race 007', 'Ball 007']

for i, (class_id, title) in enumerate(zip(example_classes, titles)):
    fname = representative[class_id]
    signal_raw = signals_dict[fname]
    signal_filtered = bandpass_filter(signal_raw, lowcut=10, highcut=5000, fs=FS)
    
    n_show = 2048
    time_ms = np.arange(n_show) / FS * 1000
    
    r, c = i // 2, i % 2
    ax = axes[r, c]
    ax.plot(time_ms, signal_raw[:n_show], alpha=0.5, linewidth=0.6, 
            color='gray', label='Raw')
    ax.plot(time_ms, signal_filtered[:n_show], linewidth=0.7, 
            color='steelblue', label='Filtered')
    ax.set_title(f'{title}', fontweight='bold')
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Raw vs Bandpass Filtered Signals', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PSD comparison: Raw vs Filtered
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

fname = representative[1]  # Inner Race fault
signal_raw = signals_dict[fname]
signal_filtered = bandpass_filter(signal_raw, lowcut=10, highcut=5000, fs=FS)

# Raw PSD
f_raw, psd_raw = welch(signal_raw, fs=FS, nperseg=2048)
f_filt, psd_filt = welch(signal_filtered, fs=FS, nperseg=2048)

axes[0].semilogy(f_raw, psd_raw, color='gray', alpha=0.7, label='Raw')
axes[0].semilogy(f_filt, psd_filt, color='steelblue', label='Filtered')
axes[0].axvline(10, color='red', linestyle=':', alpha=0.5)
axes[0].axvline(5000, color='red', linestyle=':', alpha=0.5)
axes[0].set_title('PSD: Raw vs Filtered (Inner Race)', fontweight='bold')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('PSD (g²/Hz)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Difference
axes[1].semilogy(f_raw, np.abs(psd_raw - psd_filt) + 1e-15, 
                 color='red', alpha=0.7)
axes[1].set_title('Absolute PSD Difference', fontweight='bold')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('|Δ PSD|')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Normalization Methods

We compare three normalization methods:
- **Z-score**: $(x - \mu) / \sigma$ → zero mean, unit variance
- **Min-Max**: $(x - x_{min}) / (x_{max} - x_{min})$ → scaled to [0, 1]
- **MaxAbs**: $x / |x|_{max}$ → scaled to [-1, 1]

In [ ]:
# ============================================================
# Normalization method comparison
# ============================================================
fname = representative[4]  # Outer Race fault
signal = signals_dict[fname]
signal_filtered = bandpass_filter(signal, fs=FS)
window = signal_filtered[:2048]

methods = ['zscore', 'minmax', 'maxabs']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

# Original
time_ms = np.arange(len(window)) / FS * 1000
axes[0].plot(time_ms, window, linewidth=0.7, color='gray')
axes[0].set_title(f'Original (Filtered)', fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].set_xlabel('Time (ms)')
axes[0].grid(True, alpha=0.3)
axes[0].text(0.02, 0.95, f'Mean={np.mean(window):.4f}\nStd={np.std(window):.4f}',
             transform=axes[0].transAxes, va='top', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

colors = ['steelblue', 'green', 'darkorange']
for i, (method, color) in enumerate(zip(methods, colors)):
    normalized = normalize_signal(window.copy(), method=method)
    
    ax = axes[i + 1]
    ax.plot(time_ms, normalized, linewidth=0.7, color=color)
    ax.set_title(f'{method.upper()} Normalization', fontweight='bold')
    ax.set_ylabel('Normalized Amplitude')
    ax.set_xlabel('Time (ms)')
    ax.grid(True, alpha=0.3)
    
    stats = (f'Mean={np.mean(normalized):.4f}\n'
             f'Std={np.std(normalized):.4f}\n'
             f'Range=[{np.min(normalized):.3f}, {np.max(normalized):.3f}]')
    ax.text(0.02, 0.95, stats, transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.suptitle('Normalization Methods Comparison (Outer Race 007)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✅ Z-score normalization is used in our pipeline")
print("   • Preserves signal shape and relative amplitudes")
print("   • Robust to different recording conditions")
print("   • Zero mean, unit variance → ideal for neural networks")

## 4. Windowing Strategy

We segment long signals into overlapping windows of 2048 samples (~0.17 seconds at 12 kHz).

In [ ]:
# ============================================================
# Windowing demonstration
# ============================================================
fname = representative[0]  # Normal signal
signal = signals_dict[fname][:12000]  # 1 second

# Different overlap ratios
overlaps = [0.0, 0.25, 0.5, 0.75]
window_size = 2048

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, overlap in enumerate(overlaps):
    windows, indices = create_windows(signal, window_size=window_size, overlap=overlap)
    step = int(window_size * (1 - overlap))
    
    ax = axes[i]
    time_axis = np.arange(len(signal)) / FS * 1000
    ax.plot(time_axis, signal, color='gray', linewidth=0.5, alpha=0.3)
    
    # Show windows with alternating colors
    colors_w = plt.cm.tab10(np.linspace(0, 1, min(len(windows), 10)))
    for j, (win, idx) in enumerate(zip(windows[:10], indices[:10])):
        t = np.arange(idx, idx + window_size) / FS * 1000
        ax.plot(t, win, linewidth=0.8, color=colors_w[j % len(colors_w)], alpha=0.7)
    
    ax.set_title(f'Overlap = {overlap*100:.0f}% → {len(windows)} windows (step={step})', 
                 fontweight='bold')
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Windowing Strategy (window_size={window_size}, signal=1 second)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary
print("\n📊 Windows generated per overlap:")
for overlap in overlaps:
    windows, _ = create_windows(signal, window_size=window_size, overlap=overlap)
    step = int(window_size * (1 - overlap))
    print(f"  Overlap {overlap*100:5.1f}% → Step {step:5d} → {len(windows)} windows")

print(f"\n✅ Using 50% overlap → Good balance of data augmentation and diversity")

## 5. Data Splitting Strategies — The Critical Decision

### Why splitting matters for bearing fault detection:

| Method | Description | Data Leakage Risk |
|--------|-------------|------------------|
| **Random Window** | Shuffle all windows randomly | ⚠️ HIGH — adjacent windows from same signal |
| **File-based** | Split by recording files | ✅ LOW — different recordings |
| **Time-based** | Use first 70% of each signal for train | ✅ LOW — temporal separation |
| **Hybrid** | File split + temporal within files | ✅✅ LOWEST — double protection |

In [ ]:
# ============================================================
# Compare all three splitting methods
# ============================================================
print("="*70)
print("COMPARING DATA SPLITTING STRATEGIES")
print("="*70)

split_results = {}

# 1. Time-based split
print("\n" + "-"*50)
print("1️⃣  TIME-BASED SPLIT")
print("-"*50)
X_train_t, y_train_t, X_test_t, y_test_t = time_based_split(
    signals_dict, labels_dict,
    train_time_ratio=0.7, window_size=2048, overlap=0.5, fs=FS
)
split_results['Time-based'] = (X_train_t, y_train_t, X_test_t, y_test_t)

# 2. File-based split
print("\n" + "-"*50)
print("2️⃣  FILE-BASED SPLIT")
print("-"*50)
X_train_f, y_train_f, X_test_f, y_test_f = file_based_split(
    signals_dict, labels_dict,
    train_ratio=0.75, window_size=2048, overlap=0.5, fs=FS, seed=42
)
split_results['File-based'] = (X_train_f, y_train_f, X_test_f, y_test_f)

# 3. Hybrid split
print("\n" + "-"*50)
print("3️⃣  HYBRID SPLIT")
print("-"*50)
X_train_h, y_train_h, X_test_h, y_test_h = hybrid_split(
    signals_dict, labels_dict,
    file_train_ratio=0.7, time_train_ratio=0.7,
    window_size=2048, overlap=0.5, fs=FS, seed=42
)
split_results['Hybrid'] = (X_train_h, y_train_h, X_test_h, y_test_h)

In [ ]:
# ============================================================
# Comparison table
# ============================================================
comparison_data = []
for name, (X_tr, y_tr, X_te, y_te) in split_results.items():
    comparison_data.append({
        'Method': name,
        'Train Samples': len(X_tr),
        'Test Samples': len(X_te),
        'Total': len(X_tr) + len(X_te),
        'Train %': f"{100*len(X_tr)/(len(X_tr)+len(X_te)):.1f}%",
        'Test %': f"{100*len(X_te)/(len(X_tr)+len(X_te)):.1f}%",
        'Train Classes': len(set(y_tr)),
        'Test Classes': len(set(y_te)),
    })

comp_df = pd.DataFrame(comparison_data)
print("\n📊 Split Method Comparison:")
comp_df

In [ ]:
# ============================================================
# Class distribution per split method
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (name, (X_tr, y_tr, X_te, y_te)) in enumerate(split_results.items()):
    ax = axes[i]
    
    train_counts = Counter(y_tr)
    test_counts = Counter(y_te)
    
    classes = sorted(set(y_tr) | set(y_te))
    x_pos = np.arange(len(classes))
    width = 0.35
    
    train_vals = [train_counts.get(c, 0) for c in classes]
    test_vals = [test_counts.get(c, 0) for c in classes]
    
    ax.bar(x_pos - width/2, train_vals, width, label='Train', 
           color='steelblue', alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.bar(x_pos + width/2, test_vals, width, label='Test', 
           color='coral', alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Class ID')
    ax.set_ylabel('Number of Windows')
    ax.set_title(f'{name} Split', fontweight='bold', fontsize=13)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(classes)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Class Distribution Across Split Methods', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Data Leakage Verification

We verify that no data leakage exists by checking overlapping windows between splits.

In [ ]:
# ============================================================
# Leakage check: sample-level similarity between train/test
# ============================================================
print("🔍 Data Leakage Check")
print("="*60)

for name, (X_tr, y_tr, X_te, y_te) in split_results.items():
    # Check a random subset of test windows against train windows
    n_check = min(100, len(X_te))
    check_indices = np.random.choice(len(X_te), n_check, replace=False)
    
    max_similarities = []
    for idx in check_indices:
        test_window = X_te[idx]
        # Compare against random train windows
        train_check = np.random.choice(len(X_tr), min(200, len(X_tr)), replace=False)
        similarities = np.array([
            np.corrcoef(test_window, X_tr[j])[0, 1] 
            for j in train_check
        ])
        max_similarities.append(np.max(np.abs(similarities)))
    
    max_sim = np.max(max_similarities)
    mean_sim = np.mean(max_similarities)
    
    status = "✅ PASS" if max_sim < 0.99 else "⚠️ POTENTIAL LEAKAGE"
    print(f"\n{name}:")
    print(f"  Max similarity: {max_sim:.4f}")
    print(f"  Mean max similarity: {mean_sim:.4f}")
    print(f"  Status: {status}")

In [ ]:
# ============================================================
# Visualize the time-based split on a single signal
# ============================================================
fname = representative[1]  # Inner race fault
signal = signals_dict[fname]
signal_filtered = bandpass_filter(signal, fs=FS)

split_ratio = 0.7
split_idx = int(len(signal_filtered) * split_ratio)

time_axis = np.arange(len(signal_filtered)) / FS
split_time = split_idx / FS

fig, ax = plt.subplots(figsize=(16, 5))

# Train portion
ax.plot(time_axis[:split_idx], signal_filtered[:split_idx], 
        color='steelblue', linewidth=0.3, alpha=0.7, label='Train (first 70%)')
# Test portion
ax.plot(time_axis[split_idx:], signal_filtered[split_idx:], 
        color='coral', linewidth=0.3, alpha=0.7, label='Test (last 30%)')

ax.axvline(x=split_time, color='black', linestyle='--', linewidth=2, 
           label=f'Split Point ({split_time:.1f}s)')

ax.fill_between(time_axis[:split_idx], ax.get_ylim()[0], ax.get_ylim()[1], 
                alpha=0.05, color='steelblue')
ax.fill_between(time_axis[split_idx:], ax.get_ylim()[0], ax.get_ylim()[1], 
                alpha=0.05, color='coral')

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Amplitude', fontsize=12)
ax.set_title(f'Time-Based Split — Inner Race 007 ({fname})', 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Signal length: {len(signal_filtered):,} samples ({len(signal_filtered)/FS:.1f}s)")
print(f"   Train portion: {split_idx:,} samples ({split_idx/FS:.1f}s)")
print(f"   Test portion: {len(signal_filtered)-split_idx:,} samples ({(len(signal_filtered)-split_idx)/FS:.1f}s)")

## 7. Data Augmentation

Our pipeline applies online augmentation during training:
- **Gaussian noise injection**: Adds robustness to sensor noise
- **Random amplitude scaling**: Handles varying recording conditions

In [ ]:
# ============================================================
# Augmentation effects
# ============================================================
# Use hybrid split data
X_train, y_train = X_train_h, y_train_h

# Get one sample
sample_idx = 0
original = X_train[sample_idx]

fig, axes = plt.subplots(2, 3, figsize=(18, 8))

# Original
time_ms = np.arange(len(original)) / FS * 1000
axes[0, 0].plot(time_ms, original, linewidth=0.7, color='steelblue')
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].set_ylabel('Amplitude')

# Noise augmentations
noise_levels = [0.01, 0.05, 0.1]
for i, noise_std in enumerate(noise_levels):
    augmented = original + np.random.randn(*original.shape) * noise_std
    axes[0, i].plot(time_ms, augmented, linewidth=0.7, color='coral') if i > 0 else None
    if i > 0:
        axes[0, i].set_title(f'+ Noise (σ={noise_std})', fontweight='bold')

# Actually plot in bottom row
axes[0, 1].cla()
axes[0, 1].plot(time_ms, original + np.random.randn(*original.shape) * 0.05, 
                linewidth=0.7, color='coral')
axes[0, 1].set_title('+ Noise (σ=0.05)', fontweight='bold')

axes[0, 2].cla()
axes[0, 2].plot(time_ms, original + np.random.randn(*original.shape) * 0.10, 
                linewidth=0.7, color='coral')
axes[0, 2].set_title('+ Noise (σ=0.10)', fontweight='bold')

# Scaling augmentations
scales = [0.8, 1.0, 1.2]
for i, scale in enumerate(scales):
    augmented = original * scale
    axes[1, i].plot(time_ms, augmented, linewidth=0.7, color='green')
    axes[1, i].set_title(f'Scale = {scale}x', fontweight='bold')
    axes[1, i].set_xlabel('Time (ms)')
    axes[1, i].set_ylabel('Amplitude')

for ax in axes.flatten():
    ax.grid(True, alpha=0.3)

plt.suptitle('Data Augmentation Effects', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BearingDataset & DataLoader verification
# ============================================================
print("📦 PyTorch DataLoader Verification")
print("="*50)

# Create dataset with augmentation
train_dataset = BearingDataset(X_train_h, y_train_h, augment=True)
test_dataset = BearingDataset(X_test_h, y_test_h, augment=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Get a batch
X_batch, y_batch = next(iter(train_loader))

print(f"\nTrain dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"\nBatch shape: X={X_batch.shape}, y={y_batch.shape}")
print(f"X dtype: {X_batch.dtype}")
print(f"y dtype: {y_batch.dtype}")
print(f"X range: [{X_batch.min():.3f}, {X_batch.max():.3f}]")
print(f"y values in batch: {sorted(y_batch.unique().tolist())}")
print(f"\n✅ DataLoader ready for training!")

## 8. Complete Pipeline Visualization

End-to-end: Raw signal → Filtered → Windowed → Normalized → Tensor

In [ ]:
# ============================================================
# Full pipeline visualization
# ============================================================
fname = representative[3]  # Inner Race 021 (severe)
raw_signal = signals_dict[fname]

fig, axes = plt.subplots(5, 1, figsize=(16, 20))

# Step 1: Raw signal
n_show = 12000  # 1 second
t = np.arange(n_show) / FS * 1000
axes[0].plot(t, raw_signal[:n_show], linewidth=0.5, color='gray')
axes[0].set_title('Step 1: Raw Signal (from .mat file)', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Raw Amplitude')
axes[0].grid(True, alpha=0.3)

# Step 2: Bandpass filtered
filtered = bandpass_filter(raw_signal, lowcut=10, highcut=5000, fs=FS)
axes[1].plot(t, filtered[:n_show], linewidth=0.5, color='steelblue')
axes[1].set_title('Step 2: Bandpass Filtered (10–5000 Hz, 4th order Butterworth)', 
                   fontweight='bold', fontsize=12)
axes[1].set_ylabel('Filtered Amplitude')
axes[1].grid(True, alpha=0.3)

# Step 3: Windowed
windows, indices = create_windows(filtered[:n_show], window_size=2048, overlap=0.5)
axes[2].plot(t, filtered[:n_show], linewidth=0.3, color='gray', alpha=0.3)
colors_w = plt.cm.Set2(np.linspace(0, 1, min(len(windows), 8)))
for j, (win, idx) in enumerate(zip(windows[:8], indices[:8])):
    tw = np.arange(idx, idx + 2048) / FS * 1000
    axes[2].plot(tw, win, linewidth=0.8, color=colors_w[j])
axes[2].set_title(f'Step 3: Windowing (size=2048, overlap=50%) → {len(windows)} windows', 
                   fontweight='bold', fontsize=12)
axes[2].set_ylabel('Windowed Signal')
axes[2].grid(True, alpha=0.3)

# Step 4: Normalized window
norm_window = normalize_signal(windows[0], method='zscore')
tw = np.arange(2048) / FS * 1000
axes[3].plot(tw, norm_window, linewidth=0.7, color='green')
axes[3].set_title('Step 4: Z-Score Normalized Window (μ=0, σ=1)', 
                   fontweight='bold', fontsize=12)
axes[3].set_ylabel('Normalized Value')
axes[3].grid(True, alpha=0.3)

# Step 5: Tensor
tensor = torch.FloatTensor(norm_window).unsqueeze(0).unsqueeze(0)  # [1, 1, 2048]
axes[4].plot(tw, tensor.squeeze().numpy(), linewidth=0.7, color='purple')
axes[4].set_title(f'Step 5: PyTorch Tensor — Shape: {list(tensor.shape)}', 
                   fontweight='bold', fontsize=12)
axes[4].set_xlabel('Time (ms)', fontsize=12)
axes[4].set_ylabel('Tensor Value')
axes[4].grid(True, alpha=0.3)

plt.suptitle('Complete Preprocessing Pipeline — Inner Race 021', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. Key Takeaways

### Preprocessing Pipeline Summary
```
Raw .mat file (12/48 kHz)
  │
  ├─ Downsample 48→12 kHz (files ≥ 169)
  │
  ├─ Bandpass Filter (10–5000 Hz, Butterworth 4th order, zero-phase)
  │
  ├─ Data Split (Hybrid: file-based + time-based)
  │    ├─ Train: early 70% of train-files 
  │    └─ Test: late 30% of test-files
  │
  ├─ Windowing (2048 samples, 50% overlap)
  │
  ├─ Z-Score Normalization (per window)
  │
  └─ PyTorch Tensor [batch, 1, 2048]
      + Optional augmentation (noise + scaling)
```

### Design Decisions
- **Hybrid split** chosen as default: most realistic for production deployment
- **50% overlap**: good augmentation without excessive redundancy
- **Per-window normalization**: handles varying recording conditions
- **Online augmentation**: noise injection + random scaling during training only